In [ ]:
import pandas as pd
import numpy as np
# Carga del dataset original
df = pd.read_json("../data/raw/streaming_users_dirty.json")
print("Shape inicial:", df.shape)

df.head()

## 1. Dimensiones, tipos y nulos

Antes de aplicar cualquier transformación, inspeccionamos la estructura
general del dataset: dimensiones, tipos de datos asignados automáticamente
y presencia de valores nulos por columna. Esta revisión es la base de
evidencia para todas las decisiones posteriores.

In [ ]:
# Información general: tipos de datos y cantidad de no-nulos por columna
print("Info general")
df.info()

# Conteo y porcentaje de nulos por columna
nulos = df.isnull().sum()
pct = (nulos / len(df) * 100).round(2)
print("\nNulos por columna ")
print(pd.DataFrame({"Nulos": nulos, "Porcentaje (%)": pct}))

# Detección de filas completamente duplicadas
print(f"\n=== Duplicados: {df.duplicated().sum()} ===")

# Eliminación de duplicados: un registro idéntico no aporta información nueva
df = df.drop_duplicates()
print(f"Shape tras eliminar duplicados: {df.shape}")

## 2. Estandarización de variables categóricas

Se detectaron variantes inconsistentes en `subscription_plan` (Estándar / estandar / Std),
`country` (Brasil / Brazil / brasil) y `favorite_genre` (Acción / ACCIÓN / accion / Action).
Estas diferencias impiden cualquier agrupación o análisis por categoría.
Se unifican bajo una forma canónica en minúsculas sin tildes.
No se elimina ningún registro: los valores son válidos, solo están mal escritos.

Los nulos en `favorite_genre` son un caso especial: no podemos determinar si representan
un dato no registrado o una preferencia genuinamente indefinida del usuario.
Por eso se introduce la categoría explícita `sin_definir` en lugar de imputar
un género o eliminar el registro, preservando la información sobre la ausencia del dato.

Lo mismo aplica para `subscription_plan` y `country`: cualquier valor que no pueda
mapearse se etiqueta como `sin_definir` para no perder el registro.

In [ ]:
# subscription_plan Normalizamos espacios y llevamos a minúsculas para unificar variantes
df["subscription_plan"] = df["subscription_plan"].str.strip().str.lower()

# Diccionario de mapeo: todas las variantes conocidas al valor canónico
plan_map = {
    "estandar": "estandar", "estándar": "estandar", "std": "estandar", "standard": "estandar",
    "basico": "basico", "básico": "basico", "basic": "basico", "BASICO": "basico",
    "premium": "premium", "premiun": "premium"
}

# Aplicamos el mapeo; valores no contemplados quedan como NaN
df["subscription_plan"] = df["subscription_plan"].map(plan_map)

# Verificamos si quedaron valores no mapeados
print("Nulos generados en subscription_plan:", df["subscription_plan"].isnull().sum())
print("Valores únicos subscription_plan:", df["subscription_plan"].unique())

In [ ]:
# country Normalizamos espacios y llevamos a minúsculas
df["country"] = df["country"].str.strip().str.lower()

# Diccionario de mapeo: unificamos variantes por idioma, acentuación y capitalización
country_map = {
    "argentina": "argentina", "arg": "argentina",
    "brasil": "brasil", "brazil": "brasil", "bra": "brasil",
    "chile": "chile", "chl": "chile",
    "colombia": "colombia", "col": "colombia",
    "mexico": "mexico", "méxico": "mexico", "mex": "mexico",
    "peru": "peru", "perú": "peru", "per": "peru",
    "uruguay": "uruguay", "ury": "uruguay"
}

# Aplicamos el mapeo
df["country"] = df["country"].map(country_map)

# Verificamos nulos generados por valores no contemplados en el diccionario
print("Nulos generados en country:", df["country"].isnull().sum())
print("Valores únicos country:", df["country"].unique())

In [ ]:
# favorite_genre Normalizamos espacios y llevamos a minúsculas
df["favorite_genre"] = df["favorite_genre"].str.strip().str.lower()

# Diccionario de mapeo: unificamos variantes por idioma y capitalización
genre_map = {
    "accion": "accion", "acción": "accion", "action": "accion",
    "drama": "drama",
    "crime": "crimen", "crimen": "crimen",
    "comedia": "comedia", "comedy": "comedia",
    "documental": "documental", "documentary": "documental", "doc": "documental",
    "thriller": "thriller", "thriler": "thriller",
    "romance": "romance"
}

# Aplicamos el mapeo
df["favorite_genre"] = df["favorite_genre"].map(genre_map)
print("Nulos generados en favorite_genre:", df["favorite_genre"].isnull().sum())
# Los nulos (originales y no mapeados) se etiquetan como sin_definir
# ya que no podemos distinguir entre dato ausente y preferencia indefinida
df["favorite_genre"] = df["favorite_genre"].fillna("sin_definir")
print("Valores únicos favorite_genre:", df["favorite_genre"].unique())

## 3. Tratamiento de fechas

La columna `last_login_date` presenta tres formatos mezclados (YYYY-MM-DD,
MM-DD-YYYY, YYYY/MM/DD) y fue cargada como `object`.
Se fuerza la conversión con `infer_datetime_format=True` y `errors="coerce"`,
lo que convierte automáticamente a `NaT` tanto los nulos originales como las
fechas inválidas (por ejemplo, mes 15).

Los registros con fecha futura son imposibles en el contexto real de la plataforma
y se eliminan. Los nulos restantes tampoco son recuperables: no existe criterio
válido para imputar una fecha de último acceso, por lo que esas filas se eliminan.

A partir de la fecha limpia se deriva `days_since_last_login`, variable numérica
con mayor utilidad analítica y candidata para PCA.

In [ ]:
# Convertimos la columna a datetime unificando los tres formatos detectados
# errors="coerce" transforma a NaT cualquier fecha que no pueda interpretarse
df["last_login_date"] = pd.to_datetime(
    df["last_login_date"], errors="coerce"
)

# Contamos nulos generados (originales + fechas inválidas convertidas a NaT)
print(f"Nulos tras conversión: {df['last_login_date'].isnull().sum()}")

# Fecha de referencia para detectar fechas futuras
hoy = pd.Timestamp.today().normalize()

# Una fecha de último acceso posterior a hoy es imposible en la plataforma
mask_futuras = df["last_login_date"] > hoy
print(f"Registros con fecha futura: {mask_futuras.sum()}")
df = df[~mask_futuras]

# Los nulos restantes (originales + inválidas) no pueden imputarse con ningún
# criterio válido, por lo que eliminamos esas filas
df = df.dropna(subset=["last_login_date"])
print(f"Shape tras limpiar fechas: {df.shape}")

# Derivamos days_since_last_login como diferencia en días desde hoy
# Esta variable tiene mayor utilidad numérica que la fecha en sí
df["days_since_last_login"] = (hoy - df["last_login_date"]).dt.days
print(df["days_since_last_login"].describe())

## 4. Tratamiento de `age`

Se detectaron valores de edad iguales a 0 y superiores a 120, ambos imposibles
para usuarios reales de una plataforma de streaming.
No existe criterio válido para imputar la edad de un usuario desconocido,
por lo que estos registros se eliminan directamente.

In [ ]:
# Identificamos edades fuera del rango posible para un usuario real
mask_age = (df["age"] <= 0) | (df["age"] >= 120)
print(f"Registros con age inválida: {mask_age.sum()}")

# Eliminamos esos registros ya que no son recuperables
df = df[~mask_age]
print(f"Shape tras eliminar edades imposibles: {df.shape}")

## 5. Tratamiento de `monthly_watch_time_mins`

Se aplican tres criterios en orden.

Primero, valores negativos: un tiempo de visualización negativo es físicamente
imposible y no recuperable, se eliminan.

Segundo, valores superiores a 43.200 minutos (60 × 24 × 30): superan el límite
absoluto de un mes y se eliminan. Valores altos pero dentro del rango posible,
como los cercanos a 4.000 minutos (~68 horas/mes), son outliers estadísticos
pero no errores de dato y se conservan.

Tercero, para los nulos restantes se imputa con la mediana agrupada por
`subscription_plan`. Esta decisión se justifica porque el comportamiento de
visualización varía razonablemente según el plan contratado, y la mediana
es robusta frente a los outliers conservados. Si algún registro tiene
`subscription_plan` sin_definir y no puede agruparse correctamente,
se aplica la mediana global como segundo criterio.

In [ ]:
# Eliminamos valores negativos: imposibles físicamente
mask_neg = df["monthly_watch_time_mins"] < 0
print(f"Minutos negativos: {mask_neg.sum()}")
df = df[~mask_neg]

# Eliminamos valores que superan el máximo posible en un mes (43.200 min)
mask_imposible = df["monthly_watch_time_mins"] > 43200
print(f"Minutos imposibles (>43200): {mask_imposible.sum()}")
df = df[~mask_imposible]

# Documentamos outliers válidos que se conservan
outliers_validos = df[df["monthly_watch_time_mins"] > 3000]
print(f"Outliers válidos conservados (>3000 min): {len(outliers_validos)}")

# Imputamos nulos con la mediana por subscription_plan como primer criterio
mediana_por_plan = df.groupby("subscription_plan")["monthly_watch_time_mins"].transform("median")
df["monthly_watch_time_mins"] = df["monthly_watch_time_mins"].fillna(mediana_por_plan)

# Si quedan nulos (por registros con plan sin_definir), aplicamos mediana global
mediana_global = df["monthly_watch_time_mins"].median()
df["monthly_watch_time_mins"] = df["monthly_watch_time_mins"].fillna(mediana_global)

print(f"Nulos restantes en monthly_watch_time_mins: {df['monthly_watch_time_mins'].isnull().sum()}")
print(f"Shape tras limpiar minutos: {df.shape}")

## 6. Tratamiento de `customer_support_tickets`

Se detectó un registro con 99 tickets de soporte, valor extremadamente alto
pero técnicamente posible para un usuario con problemas recurrentes.
No se elimina ni imputa: se conserva y se documenta como observación de
interés analítico.

Sin embargo, se detectaron también valores negativos, que son imposibles:
un ticket no puede tener cantidad negativa. Esos registros se eliminan.

In [ ]:
# Verificamos la distribución general de la variable
print(df["customer_support_tickets"].describe())

# Identificamos y eliminamos valores negativos, que son imposibles
mask_tickets_neg = df["customer_support_tickets"] < 0
print(f"\nRegistros con tickets negativos: {mask_tickets_neg.sum()}")
df = df[~mask_tickets_neg]

# Documentamos el outlier extremo que se conserva
print(f"Registros con más de 20 tickets (outlier válido conservado): {(df['customer_support_tickets'] > 20).sum()}")
print(f"Shape tras corregir tickets: {df.shape}")

## 7. Estado final del dataset

Verificamos que el dataset resultante no tenga nulos sin resolver,
que los tipos de datos sean correctos y que las estadísticas
descriptivas sean coherentes con los rangos esperados.

In [ ]:
# Resumen de shape, tipos y nulos tras todas las transformaciones
print("Shape final:", df.shape)

print("\nTipos de datos:")
print(df.dtypes)

print("\nNulos restantes por columna:")
print(df.isnull().sum())

print("\nEstadísticas descriptivas finales:")
df.describe()

## 8. Guardado del dataset procesado y log ETL

El dataset limpio se guarda en `data/processed/` preservando el original
intacto en `data/raw/`. El log ETL registra cada transformación con su
impacto en filas y nulos, permitiendo comparar el estado inicial y final.

In [ ]:
# Guardamos el dataset limpio en data/processed/ sin tocar el original
# en data/raw/. El log registra el estado inicial y final del dataset.
# Guardamos el dataset como JSON manteniendo la estructura de registros
df.to_json(
    "../data/processed/streaming_users_processed.json",
    orient="records",  # cada fila del df es un objeto JSON independiente
    indent=2 # indentación para que el archivo sea legible
)
print("Dataset procesado guardado.")

# Construimos el log comparando el estado inicial contra el final
log = pd.DataFrame([
    {
        "Paso": "01",
        "Descripción": "Dataset original cargado",
        "Filas": 8160,                # shape conocido de la inspección inicial
        "Nulos": 817,                 # total de nulos detectados en la inspección inicial
        "Retención (%)": 100.0        # punto de partida, sin eliminaciones
    },
    {
        "Paso": "02",
        "Descripción": "Dataset procesado final",
        "Filas": df.shape[0],                        # filas que quedaron tras toda la limpieza
        "Nulos": int(df.isnull().sum().sum()),        # nulos restantes en todo el dataset
        "Retención (%)": round(df.shape[0] / 8160 * 100, 2)  # porcentaje de filas conservadas
    }
])

# Guardamos el log como CSV en la carpeta logs/
log.to_csv(
    "../logs/pipeline_log.csv",
    index=False  # no incluimos el índice de pandas como columna extra
)
print("Log ETL guardado.")

# Mostramos el log en pantalla para verificar
log